# DEEPX Tutorial 04 - DX-STREAM Workflow

This tutorial introduces DX-STREAM v3.1.x and shows how to build end-to-end Vision AI pipelines on a DEEPX NPU.

You will:

- Understand where DX-STREAM fits in the DEEPX SDK
- Build a pipeline with `DxPreprocess → DxInfer → DxPostprocess → DxTracker → DxOsd`
- Learn the DX multi-stream domain and selector boundaries
- Use inference backends, scaling, conversion, and message-broker elements
- Adapt a custom postprocess library for a custom AI model
- Add a metadata-driven people-count overlay
- Inspect metadata and diagnose GStreamer pipelines


## 0. DX-STREAM Overview

DX-STREAM is a collection of GStreamer elements for Vision AI pipelines on DEEPX NPUs.

```text
Source → Decode → DxPreprocess → DxInfer → DxPostprocess → DxTracker → DxOsd → Sink
```

![DX-STREAM pipeline](assets/dx-stream-pipeline.png)

| Element | Purpose |
|---|---|
| `dxpreprocess` | Resizes, crops, and converts frames into model input tensors |
| `dxinfer` | Runs a compiled `.dxnn` model |
| `dxpostprocess` | Decodes output tensors and writes inference metadata |
| `dxtracker` | Assigns persistent IDs to detected objects |
| `dxosd` | Draws boxes, labels, poses, and segmentation results |
| `dxgather` | Merges branches created from the same source |
| `dxinputselector` | Merges input streams into the DX multi-stream domain |
| `dxoutputselector` | Splits the DX multi-stream domain by stream ID |
| `dxrate` | Controls output frame rate |
| `dxmsgconv` | Converts inference metadata into structured messages |
| `dxmsgbroker` | Publishes messages to MQTT or Kafka |
| `dxscale` | Changes frame resolution |
| `dxconvert` | Changes frame color format |

For the complete element and property reference, see the DX-STREAM User Manual available from the [DEEPX Developer Portal](https://developer.deepx.ai/download/?id=583). Login is required.


### What changed in DX-STREAM v3.1.x

DX-STREAM v3.1.0 introduced the main runtime changes covered in this tutorial. Version 3.1.1 primarily improves the documentation.

- A unified multi-stream domain based on `application/x-dxvideoraw`
- Per-stream lifecycle and timeline preservation in a shared processing chain
- The `dxinfer` backend abstraction and asynchronous Put/Get processing
- `dxmsgconv include-frame` for Base64-encoded JPEG frames
- Updated custom-library and metadata APIs
- Improved latency reporting, FLUSH recovery, caps negotiation, and error reporting

The available backend and element properties depend on how DX-STREAM was built. Use `gst-inspect-1.0` to check the installed build.


## 1. Prerequisites

This tutorial assumes that the DEEPX SDK was installed in Tutorial 01 and that the SDK location is stored in `dx-tutorials/config.json`.


### 1.1 Load the SDK paths

The following cell derives `DX_STREAM_DIR` and the other SDK paths from `DX_ALL_SUITE_DIR`, then changes the notebook working directory to the DX-STREAM repository root.


In [ ]:
# Load all SDK paths from dx-tutorials/config.json.
import os

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")

%run "$root_path/tutorial_paths.py"
print_tutorial_paths()
%cd $DX_STREAM_DIR


### 1.2 (Optional) Intel GPU prerequisite

For Intel x86_64 iGPU/dGPU systems only, follow [`docs/intel_gpu.md`](https://github.com/DEEPX-AI/dx-tutorials/blob/main/docs/intel_gpu.md).


### 1.3 Download the sample resources

`setup.sh` downloads the models and videos required by the bundled pipelines into `dx_stream/samples/`. The current model list contains models compiled with recent DX-COM releases.


In [ ]:
# Download sample models and videos.
!./setup.sh


### 1.4 Verify the installed DX-STREAM elements

The plugin should expose the following 13 elements:

`dxconvert`, `dxgather`, `dxinfer`, `dxinputselector`, `dxmsgbroker`, `dxmsgconv`, `dxosd`, `dxoutputselector`, `dxpostprocess`, `dxpreprocess`, `dxrate`, `dxscale`, and `dxtracker`.


In [ ]:
!gst-inspect-1.0 dxstream || echo "DX-STREAM plugin not found"


## 2. Quick Start

### 2.1 Run a YOLO26n object-detection pipeline

This pipeline reads a video, preprocesses each frame, runs YOLO26n, decodes the results, and displays the annotated output.

The pipeline opens a native display window and keeps the code cell busy until the pipeline stops. For the most reliable GUI behavior, open **File > New > Terminal**, change to the configured DX-STREAM directory, and run the same `gst-launch-1.0` command there:

```bash
cd <DX_STREAM_DIR>
# Copy and run the gst-launch-1.0 command from the next cell.
```

Press `Ctrl+C` in the terminal to stop the pipeline. If you run it in the Notebook instead, use the notebook stop button (`■`).


In [ ]:
VIDEO_SRC = "dx_stream/samples/videos/doughnut.mp4"
MODEL_PATH = "dx_stream/samples/models/yolo26-n_640x640.dxnn"

!gst-launch-1.0 filesrc location=$VIDEO_SRC ! decodebin ! \
    dxpreprocess \
        preprocess-id=1 \
        resize-width=640 \
        resize-height=640 ! \
    queue max-size-buffers=1 ! \
    dxinfer \
        preprocess-id=1 \
        inference-id=1 \
        model-path=$MODEL_PATH \
        backend=auto ! \
    queue max-size-buffers=1 ! \
    dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolo26od.so \
        function-name=PostProcess ! \
    queue max-size-buffers=1 ! \
    dxosd ! \
    dxconvert ! fpsdisplaysink sync=false


### 2.2 Inspect the core elements

`gst-inspect-1.0` shows the pad templates, properties, defaults, and supported enum values for the installed version.


#### 2.2.1 DxPreprocess

```bash
dxpreprocess preprocess-id=1 resize-width=640 resize-height=640
```

`preprocess-id` identifies the generated input tensor. A downstream `dxinfer` must reference the same ID.


In [ ]:
!gst-inspect-1.0 dxpreprocess


#### 2.2.2 DxInfer and inference backends

```bash
dxinfer preprocess-id=1 inference-id=1 model-path=/path/to/model.dxnn backend=auto
```

- `auto`: selects an available compiled backend
- `dxrt`: uses the DEEPX Runtime backend
- `dxvnpu`: uses the VNPU backend when DX-STREAM was built with VNPU support

`inference-id` identifies the output tensors consumed by `dxpostprocess`. DX-STREAM v3.1.x uses an asynchronous Put/Get backend interface internally; the pipeline syntax remains unchanged.


In [ ]:
!gst-inspect-1.0 dxinfer


#### 2.2.3 DxPostprocess

```bash
dxpostprocess inference-id=1 \
  library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolo26od.so \
  function-name=PostProcess
```

The `inference-id` must match the upstream inference output. The shared library and function must match the model architecture.


In [ ]:
!gst-inspect-1.0 dxpostprocess


#### 2.2.4 DxOsd

DxOsd reads inference metadata and overlays visual results on the video frame.


In [ ]:
!gst-inspect-1.0 dxosd


### 2.3 Scale and convert video frames

`dxscale` changes resolution, while `dxconvert` changes color format. On supported platforms, an accelerated kernel is selected automatically; unsupported combinations fall back to a software implementation.

```bash
... ! dxscale width=640 height=480 ! \
      dxconvert ! video/x-raw,format=RGB ! ...
```

`dxconvert` does not resize frames, and `dxscale` does not select the final color format.


In [ ]:
!gst-inspect-1.0 dxscale


In [ ]:
!gst-inspect-1.0 dxconvert


## 3. Run the Bundled Demo Pipelines

`run_demo.sh` provides 11 selections: `0` through `9`, plus `-` for Secondary Mode. If no selection is received within ten seconds, it runs option `0`.

These demos open native GUI windows and keep a Notebook cell busy until the selected pipeline exits. The recommended method is to open **File > New > Terminal** and run:

```bash
cd <DX_STREAM_DIR>
./run_demo.sh
```

Select the desired menu item in the terminal and press `Ctrl+C` to stop it. The following cells remain available for classroom demonstrations, but they intentionally block while their GUI pipeline is running.


In [ ]:
# Display the menu implemented by the installed run_demo.sh.
!sed -n '/echo "0:/,/read -t/p' run_demo.sh


Stop a running pipeline with the notebook stop button (`■`).

### 3.1 Object Detection - YOLO26n

![Single object-detection pipeline](assets/pipline-single-detection.png)


In [ ]:
!./run_demo.sh <<< 0


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/object_detection/run_yolo26n.sh


### 3.2 Object Detection with PPU - YOLOv5s


In [ ]:
!./run_demo.sh <<< 1


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/object_detection/run_YoloV5S_PPU.sh


### 3.3 Face Detection


In [ ]:
!./run_demo.sh <<< 2


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/face_detection/run_YOLOv5s_Face.sh


The next example uses the SCRFD500M model with PPU output.


In [ ]:
!./run_demo.sh <<< 3


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/face_detection/run_SCRFD500M_PPU.sh


#### Use a camera source

The first example requests raw video. The second requests MJPEG, which is not supported by every USB camera. Adjust the device and caps to match `v4l2-ctl --list-formats-ext --device=/dev/video0`.


In [ ]:
# Camera input using a raw video format.
!gst-launch-1.0 \
  v4l2src device=/dev/video0 do-timestamp=true ! \
  videoconvert ! video/x-raw,width=1920,height=1080 ! queue ! \
  dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
  queue max-size-buffers=1 ! \
  dxinfer \
    preprocess-id=1 inference-id=1 \
    model-path=dx_stream/samples/models/yolov5-s-face_640x640.dxnn \
    backend=auto ! \
  queue max-size-buffers=1 ! \
  dxpostprocess \
    inference-id=1 \
    library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov5s_face.so \
    function-name=PostProcess ! \
  queue max-size-buffers=1 ! \
  dxosd ! videoconvert ! fpsdisplaysink sync=false


In [ ]:
# Camera input using MJPEG.
!gst-launch-1.0 \
  v4l2src device=/dev/video0 do-timestamp=true ! \
  image/jpeg,width=1920,height=1080,framerate=30/1 ! \
  jpegdec ! videoconvert ! queue ! \
  dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
  queue max-size-buffers=1 ! \
  dxinfer \
    preprocess-id=1 inference-id=1 \
    model-path=dx_stream/samples/models/yolov5-s-face_640x640.dxnn \
    backend=auto ! \
  queue max-size-buffers=1 ! \
  dxpostprocess \
    inference-id=1 \
    library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov5s_face.so \
    function-name=PostProcess ! \
  queue max-size-buffers=1 ! \
  dxosd ! videoconvert ! fpsdisplaysink sync=false


### 3.4 Pose Estimation


In [ ]:
!./run_demo.sh <<< 4


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/pose_estimation/run_yolo26n-pose.sh


The next example uses YOLOv5 Pose with PPU output.


In [ ]:
!./run_demo.sh <<< 5


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/pose_estimation/run_YOLOV5Pose_PPU.sh


### 3.5 Instance Segmentation - YOLO26n-Seg

Option `6` runs the segmentation demo. The script is under `instance_segmentation` in the current SDK layout.


In [ ]:
!./run_demo.sh <<< 6


In [ ]:
!sed -n '1,240p' dx_stream/pipelines/single_network/instance_segmentation/run_yolo26n-seg.sh


### 3.6 Multi-Object Tracking

Option `7` runs object detection followed by tracking.

![Single tracking pipeline](assets/pipline-single-tracking.png)


In [ ]:
!./run_demo.sh <<< 7


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/tracking/run_multi_object_tracker.sh


### 3.7 Multi-Channel Object Detection

Option `8` runs four independent inference branches and composites their output.

![Multi-channel pipeline](assets/pipeline-multi-stream.png)


In [ ]:
!./run_demo.sh <<< 8


In [ ]:
!sed -n '1,280p' dx_stream/pipelines/multi_stream/run_multi_stream.sh


### 3.8 The DX multi-stream domain

DX-STREAM v3.1.0 introduced the `application/x-dxvideoraw` caps domain. It lets multiple streams share one processing chain while preserving the identity, dimensions, format, metadata, and timeline of each stream.

```text
N × video/x-raw
       ↓
dxinputselector                 domain entry
       ↓ application/x-dxvideoraw
dxpreprocess → dxinfer → dxpostprocess → dxosd
       ↓ application/x-dxvideoraw
dxoutputselector                domain exit
       ↓
N × video/x-raw
```

`dxinputselector` assigns a stream ID and creates `DXFrameMeta` when needed. `dxoutputselector` routes each buffer and restores the corresponding per-stream events.

The domain preserves per-stream `STREAM_START`, `CAPS`, `SEGMENT`, `TAG`, `EOS`, and `GAP` events. A slow stream can therefore handle its own QoS without throttling every input.


#### Element placement rules

| Element | Inside `application/x-dxvideoraw` | Placement note |
|---|---:|---|
| `dxpreprocess`, `dxinfer`, `dxpostprocess`, `dxtracker`, `dxosd`, `dxrate` | Yes | These elements operate in single-stream or domain mode |
| `dxscale`, `dxconvert` | No | Place them before `dxinputselector` or after `dxoutputselector` |
| `dxgather` | No | It merges branches from one source, not different stream IDs |
| Standard elements such as `videoconvert`, `videoscale`, and `compositor` | No | They accept `video/x-raw`, not `application/x-dxvideoraw` |
| `dxinputselector`, `dxoutputselector` | Boundary only | They enter and exit the domain |

Incorrect placement fails during caps negotiation instead of producing an ambiguous runtime result.


#### Share one inference chain

When all channels use the same model, a single shared inference chain avoids loading the model once per channel and reduces NPU memory usage.

<img src="assets/pipeline-multi-stream-single-infer.png" style="max-width: 1400px;">

The current example uses `run_multi_stream_selector.sh`.


In [ ]:
!dx_stream/pipelines/multi_stream/run_multi_stream_selector.sh


In [ ]:
!sed -n '1,340p' dx_stream/pipelines/multi_stream/run_multi_stream_selector.sh


### 3.9 Multi-Channel RTSP

Option `9` demonstrates multiple RTSP sources. Live pipelines benefit from the corrected v3.1.x latency and QoS reporting.


In [ ]:
!./run_demo.sh <<< 9


In [ ]:
!sed -n '1,300p' dx_stream/pipelines/rtsp/run_RTSP.sh


### 3.10 Secondary Mode

![Secondary-mode pipeline](assets/pipeline-secondary.png)

- **Primary Mode** preprocesses and infers the entire frame. Postprocessing normally creates new `DXObjectMeta` objects.
- **Secondary Mode** preprocesses detected object regions. Postprocessing updates or enriches the existing object metadata.


In [ ]:
!./run_demo.sh <<< -


In [ ]:
!sed -n '1,320p' dx_stream/pipelines/secondary_mode/run_secondary_mode.sh


### 3.11 Publish inference results to MQTT or Kafka

`dxmsgconv` converts inference metadata with a custom message-conversion library. `dxmsgbroker` publishes the payload to an MQTT or Kafka broker.

```text
... → dxpostprocess → dxmsgconv → dxmsgbroker
```

In v3.1.x, `include-frame=true` adds the current frame as a Base64-encoded JPEG to the message. This increases JPEG encoding work, message size, and network traffic, so enable it only when the consumer needs the image.

```bash
dxmsgconv library-file-path=/path/to/libmessage_convert.so include-frame=true ! \
dxmsgbroker broker-name=mqtt conn-info=localhost:1883 topic=test
```

The broker must be running before the pipeline starts. The bundled scripts provide complete MQTT and Kafka examples but are not part of the interactive `run_demo.sh` menu.


In [ ]:
!gst-inspect-1.0 dxmsgconv


In [ ]:
!gst-inspect-1.0 dxmsgbroker


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/broker/run_dxmsgbroker_mqtt.sh


In [ ]:
!sed -n '1,260p' dx_stream/pipelines/broker/run_dxmsgbroker_kafka.sh


## 4. Writing Your Own Application

This section integrates the Forklift and Worker detector created in Tutorial 03. The model uses the YOLOv7 output format but has two classes instead of the 80 COCO classes, so its postprocess configuration must be adapted.

![Custom pipeline](assets/custom-pipeline.png)

<img src="assets/detection-goal.jpg" style="max-width: 1400px;">


### 4.1 DX-STREAM v3.1.x custom-library API

Custom preprocess and postprocess libraries written for older versions must be updated as follows:

- `DXFrameMeta::_buf` was removed to avoid a circular buffer reference.
- The first custom-function argument is now `GstBuffer *buf`.
- Create object metadata with `dx_acquire_obj_meta_from_pool()`.
- Attach a new object with `dx_add_obj_meta_to_frame()`.
- In Primary Mode, postprocessing creates result objects.
- In Secondary Mode, postprocessing updates the object passed in `object_meta`.

The current YoloV7 library already uses the v3.1.x function form:

```cpp
extern "C" void PostProcess(
    GstBuffer* buf,
    std::vector<dxs::DXTensor> network_output,
    DXFrameMeta* frame_meta,
    DXObjectMeta* object_meta)
{
    DXObjectMeta* result = dx_acquire_obj_meta_from_pool();

    // Decode tensors and populate result.

    dx_add_obj_meta_to_frame(frame_meta, result);
}
```


### 4.2 Metadata hierarchy

Inference results travel with the `GstBuffer` instead of through a separate side channel.

```text
GstBuffer
└── DXFrameMeta
    ├── stream ID, width, height, format, ROI
    ├── input tensors  (preprocess ID → tensors)
    ├── output tensors (inference ID → tensors)
    ├── frame-level classification or segmentation
    ├── DXObjectMeta[]
    │   ├── label, confidence, box, tracking ID
    │   ├── keypoints, features, OBB, face, segmentation
    │   └── DXUserMeta[]
    └── DXUserMeta[]
```

`DXUserMeta` can attach application-specific data to a frame or object. A user-meta implementation must provide both copy and release functions so metadata remains valid when GStreamer copies or releases buffers.


### 4.3 Download the custom DXNN model


In [ ]:
CUSTOM_MODEL_URL = "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/yolov7-forklift-person.dxnn"
!wget -nc "$CUSTOM_MODEL_URL"


### 4.4 Adapt the YoloV7 postprocess configuration

The output decoder already matches the model architecture. Only the number and names of classes must change.


In [ ]:
%%writefile dx_stream_update.diff
diff --git a/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp b/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
--- a/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
+++ b/dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp
@@ -55,22 +55,12 @@ struct YoloConfig {
     // Detection thresholds (adjust based on your requirements)
     float conf_threshold = 0.25f;    // Minimum confidence for detection
     float nms_threshold = 0.4f;      // IoU threshold for NMS
-    
+
     // Number of classes in your dataset
-    int num_classes = 80;
-    
+    int num_classes = 2;
+
     // COCO dataset class names (modify for your dataset)
     std::vector<std::string> class_names = {
-        "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
-        "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench",
-        "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
-        "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
-        "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
-        "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
-        "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
-        "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
-        "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
-        "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
-        "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
+        "Forklift", "Worker"
     };
 };


### 4.5 Apply the patch

The cell below is safe to run more than once. It applies the patch when needed, reports success when the same patch is already present, and stops without changing the file when the source has conflicting modifications. It does not reset or discard existing work.


In [ ]:
%%bash
PATCH_FILE="dx_stream_update.diff"
TARGET_FILE="dx_stream/custom_library/postprocess_library/YoloV7/postprocess.cpp"

if git apply --check "$PATCH_FILE" 2>/dev/null; then
    git apply --whitespace=fix "$PATCH_FILE"
    echo "Patch applied successfully: $TARGET_FILE"
elif git apply --reverse --check "$PATCH_FILE" 2>/dev/null; then
    echo "Patch is already applied. No changes were needed: $TARGET_FILE"
else
    echo "ERROR: The patch is neither applicable nor already fully applied." >&2
    echo "The target may contain conflicting or partially applied changes: $TARGET_FILE" >&2
    echo "Review it with: git diff -- $TARGET_FILE" >&2
    exit 1
fi


### 4.6 Rebuild DX-STREAM

The build installs the updated custom postprocess library under the configured installation prefix.


In [ ]:
!source ../venv-dx-runtime/bin/activate && ./build.sh


### 4.7 Create the inference configuration

`dxinfer` properties can be supplied directly in the pipeline or through a JSON file.

| JSON field | Meaning |
|---|---|
| `preprocess_id` | Selects the input tensors created by the `dxpreprocess` element with the same ID |
| `inference_id` | Labels the model output tensors; `dxpostprocess` must use the same ID |
| `model_path` | Path to the compiled `.dxnn` model; a relative path is resolved from the process working directory |
| `backend` | Selects `auto`, `dxrt`, or a compiled optional backend |

Using explicit IDs is important in pipelines containing multiple preprocess, inference, or postprocess elements.


In [ ]:
import json

yolov7_custom = {
    "preprocess_id": 1,
    "inference_id": 1,
    "model_path": "./yolov7-forklift-person.dxnn",
    "backend": "auto",
}

with open("yolov7-forklift-person.json", "w", encoding="utf-8") as config_file:
    json.dump(yolov7_custom, config_file, indent=2)

print(json.dumps(yolov7_custom, indent=2))


### 4.8 Inspect the compiled model

`dxparse` shows the model input and output tensor structure. Confirm that the tensor shapes match the assumptions in the custom postprocessor.


In [ ]:
!dxparse -m yolov7-forklift-person.dxnn -v


### 4.9 Download a test video


In [ ]:
CUSTOM_VIDEO_URL = "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/forklift-worker.mp4"
!wget -nc "$CUSTOM_VIDEO_URL"


### 4.10 Run the custom pipeline

The IDs form the following relationship:

```text
dxpreprocess(preprocess-id=1)
        ↓
dxinfer(preprocess_id=1, inference_id=1)
        ↓
dxpostprocess(inference-id=1)
```


In [ ]:
VIDEO_SRC = "forklift-worker.mp4"

!gst-launch-1.0 filesrc location=$VIDEO_SRC ! decodebin ! \
    dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 ! \
    queue max-size-buffers=1 ! \
    dxinfer config-file-path=yolov7-forklift-person.json ! \
    queue max-size-buffers=1 ! \
    dxpostprocess \
        inference-id=1 \
        library-file-path=/usr/local/share/gstdxstream/lib/libpostprocess_yolov7.so \
        function-name=PostProcess ! \
    queue max-size-buffers=1 ! \
    dxosd ! videoconvert ! fpsdisplaysink sync=false


## 5. Build a YOLO26s People-Counter Application

This exercise first implements people counting in Python so that each integration point is easy to inspect. It runs YOLO26s object detection, reads the detection metadata attached to every video frame, counts COCO class `0` (`person`), and updates a text overlay.

By the end of this section, you will understand:

- where `DXObjectMeta` is defined and created,
- how detection metadata travels with a `GstBuffer`,
- how `pydxs` exposes C++ metadata to Python,
- how a Python string becomes a running GStreamer pipeline, and
- why the pad-probe callback and display update use different execution contexts.


### 5.1 Prepare the YOLO26s model and video

`yolo26-s_640x640.dxnn` is downloaded directly because the bundled DX-STREAM sample list currently includes YOLO26n but not YOLO26s. The snowboard video is selected because it contains multiple people.


In [ ]:
from pathlib import Path
import subprocess

YOLO26S_MODEL_URL = "https://sdk.deepx.ai/modelzoo/dxnn/2_4_0/yolo26-s_640x640.dxnn"
YOLO26S_MODEL = DX_STREAM_DIR / "dx_stream/samples/models/yolo26-s_640x640.dxnn"
PEOPLE_VIDEO = DX_STREAM_DIR / "dx_stream/samples/videos/snowboard.mp4"

YOLO26S_MODEL.parent.mkdir(parents=True, exist_ok=True)

if not YOLO26S_MODEL.is_file():
    partial_model = YOLO26S_MODEL.with_suffix(YOLO26S_MODEL.suffix + ".part")
    subprocess.run(
        [
            "wget",
            "--continue",
            f"--output-document={partial_model}",
            YOLO26S_MODEL_URL,
        ],
        check=True,
    )
    partial_model.replace(YOLO26S_MODEL)
else:
    print(f"Model already exists: {YOLO26S_MODEL}")

if not PEOPLE_VIDEO.is_file():
    raise FileNotFoundError(
        f"Sample video not found: {PEOPLE_VIDEO}\n"
        "Run ./setup.sh from the DX-STREAM directory first."
    )

print(f"Model: {YOLO26S_MODEL}")
print(f"Video: {PEOPLE_VIDEO}")


### 5.2 Understand the detection metadata

DX-STREAM keeps pixels and inference results together without writing detection data into the image itself. One decoded video frame is carried by a `GstBuffer`; DX-STREAM attaches one `DXFrameMeta` to that buffer, and the frame metadata owns a list of detected objects.

```text
GstBuffer — one video frame moving through the pipeline
│
├── Video memory / pixels
│
└── DXFrameMeta — frame-level GstMeta
    ├── stream_id, width, height, format, frame_rate, ROI, ...
    │
    └── object_meta_list
        ├── DXObjectMeta #0
        │   ├── label          = 0
        │   ├── label_name     = "person"
        │   ├── confidence     = 0.92
        │   └── box            = [x1, y1, x2, y2]
        ├── DXObjectMeta #1
        └── ...
```

This relationship is important: the Python callback does not run inference again and does not parse the image. It only reads object metadata already produced by `dxpostprocess`.


#### 5.2.1 Where `DXObjectMeta` is defined and created

The SDK defines the structures in these headers relative to `DX_STREAM_DIR`:

| Purpose | SDK source |
|---|---|
| Frame metadata and its object list | `gst-dxstream-plugin/metadata/gst-dxframemeta.hpp` |
| Per-object detection metadata | `gst-dxstream-plugin/metadata/gst-dxobjectmeta.hpp` |
| Python bindings for both structures | `bindings/python/pydxs/src/metadata_binding.cpp` |

For primary object detection, `dxpostprocess` calls the configured C++ `PostProcess` function. That function converts YOLO tensor output into objects using this lifecycle:

```cpp
DXObjectMeta* object = dx_acquire_obj_meta_from_pool();
object->_label       = class_id;
object->_label_name  = class_name;
object->_confidence  = score;
object->_box         = {x1, y1, x2, y2};
dx_add_obj_meta_to_frame(frame_meta, object);
```

The result is attached to the current frame, so downstream elements see the same detections:

```text
dxinfer                 dxpostprocess                         downstream
tensor output ────────▶ create DXObjectMeta ────────┬──────▶ dxosd draws boxes
                                                    └──────▶ Python probe counts people
```

DX-STREAM manages the metadata lifetime with the buffer. If an element creates a new buffer and copies `DXFrameMeta`, its object metadata is copied as well. Application code should nevertheless treat a Python metadata reference as valid only while processing the current buffer callback; do not save it for later use.


#### 5.2.2 How C++ metadata appears in Python

`pydxs` maps the C++ members to Python-friendly properties:

| C++ field | Python property | Meaning in this exercise |
|---|---|---|
| `DXFrameMeta::_object_meta_list` | `frame_meta.object_meta_list` or iteration over `frame_meta` | All detections in the current frame |
| `DXObjectMeta::_label` | `obj_meta.label` | COCO class ID; `0` means `person` |
| `DXObjectMeta::_label_name` | `obj_meta.label_name` | Human-readable class name |
| `DXObjectMeta::_confidence` | `obj_meta.confidence` | Detection confidence |
| `DXObjectMeta::_box` | `obj_meta.box` | `[left, top, right, bottom]` in frame coordinates |
| `DXObjectMeta::_track_id` | `obj_meta.track_id` | Tracker ID; not assigned in this pipeline |

The callback receives the same `Gst.Buffer` that is about to leave `dxpostprocess`:

```python
buffer = info.get_buffer()
frame_meta = pydxs.dx_get_frame_meta(hash(buffer))

people_count = sum(
    1 for obj_meta in frame_meta
    if obj_meta.label == PERSON_CLASS_ID
)
```

`hash(buffer)` supplies the native `GstBuffer` address expected by the binding. `frame_meta` is iterable because `pydxs` exposes `_object_meta_list` through Python's iterator protocol.

The count is therefore the number of accepted `person` detection boxes in the **current frame**. The Python code does not apply another confidence threshold. Confidence filtering and duplicate suppression, if used, must happen in the postprocessor before objects are attached. This is not a unique visitor count; tracking is required to maintain identities over time.


### 5.3 Understand how Python embeds the GStreamer pipeline

The application does not invoke `gst-launch-1.0` as a subprocess. Instead, `pipeline_description` contains the same GStreamer launch syntax as a Python formatted string, and `Gst.parse_launch()` converts that text into real GStreamer elements connected by pads.

```text
Python application
│
├── pipeline_description = f''' ... '''
│       │
│       └── Gst.parse_launch(description)
│                    │
│                    ▼
│    ┌────────┐  ┌──────────┐  ┌───────┐  ┌─────────────┐  ┌───────┐
└───▶│ source │─▶│preprocess│─▶│ infer │─▶│ postprocess │─▶│  osd  │─▶ display
     └────────┘  └──────────┘  └───────┘  └──────┬──────┘  └───────┘
                                                  │
                                    source-pad BUFFER probe
                                                  │
                                                  ▼
                                    read metadata → count people
                                                  │
                                                  ▼
                                      update `textoverlay` text
```

The probe observes the buffer; it is not another pipeline branch and does not copy the frame.

| Pipeline part | Responsibility |
|---|---|
| `urisourcebin ! decodebin` | Read the video and decode compressed frames |
| `dxpreprocess` | Resize and prepare the frame for the model |
| `dxinfer` | Run YOLO26s and attach its output tensors |
| `dxpostprocess name=detector_postprocess` | Decode tensors and attach `DXObjectMeta` entries |
| `dxosd` | Read object metadata and draw boxes and labels |
| `dxconvert ! videoconvert` | Prepare the video format for the standard overlay/display path |
| `textoverlay name=people_overlay` | Display the count controlled by Python |
| `fpsdisplaysink` | Present frames and measure display FPS |


#### 5.3.1 From a pipeline string to named Python objects

Three lines connect the pipeline text to the application logic:

```python
self.pipeline = Gst.parse_launch(pipeline_description)
self.people_overlay = self.pipeline.get_by_name("people_overlay")
postprocess = self.pipeline.get_by_name("detector_postprocess")
```

The names come from properties inside the launch string:

```text
dxpostprocess name=detector_postprocess ...
textoverlay   name=people_overlay ...
```

The application then attaches a callback to the postprocessor's source pad:

```python
src_pad = postprocess.get_static_pad("src")
src_pad.add_probe(Gst.PadProbeType.BUFFER, self._postprocess_probe)
```

Every output buffer from `dxpostprocess` invokes `_postprocess_probe` before continuing to `dxosd`. Returning `Gst.PadProbeReturn.OK` allows that same buffer to continue downstream.


#### 5.3.2 Streaming callback and GLib main loop

GStreamer calls the pad probe from a streaming thread. A slow callback would delay every following frame, so the probe only retrieves metadata, counts objects, and schedules the display update.

```text
GStreamer streaming thread                    GLib main-loop thread
──────────────────────────                    ─────────────────────
buffer reaches postprocess.src
          │
          ▼
_postprocess_probe()
  ├── get GstBuffer
  ├── get DXFrameMeta
  ├── count label == 0
  └── GLib.idle_add(_update_overlays, count) ───────────────┐
          │                                                 ▼
          └── return OK                           _update_overlays()
                    │                               └── set text property
                    ▼
             buffer continues                     UI remains responsive
```

The bus also reports pipeline-wide events to the main loop:

```text
GStreamer bus ── ERROR ──▶ print details and stop
              └─ EOS   ──▶ stop the main loop
```

`pipeline.set_state(Gst.State.PLAYING)` starts data flow, while `GLib.MainLoop().run()` keeps the Python application alive so it can process idle callbacks and bus messages.


### 5.4 Create the Python application

The complete application below combines the pipeline, metadata probe, overlay update, bus handling, and command-line arguments described above.


In [ ]:
%%writefile dx_stream/apps/yolo26_people_counter.py
#!/usr/bin/env python3
"""Run YOLO26s and display the number of detected people."""

from __future__ import annotations

import argparse
import signal
from pathlib import Path

import gi

gi.require_version("Gst", "1.0")
gi.require_version("GLib", "2.0")
from gi.repository import GLib, Gst

import pydxs


PERSON_CLASS_ID = 0


class PeopleCounterApplication:
    def __init__(self, video_path: Path, model_path: Path, postprocess_library: Path):
        self.video_path = video_path.expanduser().resolve()
        self.model_path = model_path.expanduser().resolve()
        self.postprocess_library = postprocess_library.expanduser().resolve()

        for path in (self.video_path, self.model_path, self.postprocess_library):
            if not path.is_file():
                raise FileNotFoundError(path)

        Gst.init(None)
        self.loop = GLib.MainLoop()
        self.exit_code = 0
        self.last_requested_count = None

        pipeline_description = f"""
            urisourcebin uri="{self.video_path.as_uri()}" ! decodebin !
            dxpreprocess preprocess-id=1 resize-width=640 resize-height=640 !
            queue max-size-buffers=1 !
            dxinfer preprocess-id=1 inference-id=1
                model-path="{self.model_path}" backend=auto !
            queue max-size-buffers=1 !
            dxpostprocess name=detector_postprocess inference-id=1
                library-file-path="{self.postprocess_library}"
                function-name=PostProcess !
            queue max-size-buffers=1 !
            dxosd ! dxconvert ! videoconvert !
            textoverlay name=people_overlay
                text="People: 0" halignment=right valignment=top
                xpad=24 ypad=24 font-desc="Sans Bold 24"
                shaded-background=true !
            fpsdisplaysink sync=true text-overlay=false
        """

        self.pipeline = Gst.parse_launch(pipeline_description)
        self.people_overlay = self.pipeline.get_by_name("people_overlay")
        postprocess = self.pipeline.get_by_name("detector_postprocess")
        if self.people_overlay is None or postprocess is None:
            raise RuntimeError("Failed to create required GStreamer elements")

        src_pad = postprocess.get_static_pad("src")
        if src_pad is None:
            raise RuntimeError("dxpostprocess source pad was not found")
        src_pad.add_probe(Gst.PadProbeType.BUFFER, self._postprocess_probe)

        bus = self.pipeline.get_bus()
        bus.add_signal_watch()
        bus.connect("message", self._on_bus_message)

    def _postprocess_probe(self, _pad, info):
        buffer = info.get_buffer()
        if buffer is None:
            return Gst.PadProbeReturn.OK

        frame_meta = pydxs.dx_get_frame_meta(hash(buffer))
        if frame_meta is None:
            return Gst.PadProbeReturn.OK

        people_count = sum(
            1 for obj_meta in frame_meta if obj_meta.label == PERSON_CLASS_ID
        )
        if people_count != self.last_requested_count:
            self.last_requested_count = people_count
            GLib.idle_add(self._update_overlays, people_count)

        return Gst.PadProbeReturn.OK

    def _update_overlays(self, people_count: int):
        self.people_overlay.set_property("text", f"People: {people_count}")
        return GLib.SOURCE_REMOVE

    def _on_bus_message(self, _bus, message):
        if message.type == Gst.MessageType.ERROR:
            error, debug = message.parse_error()
            print(f"GStreamer error: {error}")
            if debug:
                print(f"Debug details: {debug}")
            self.exit_code = 1
            self.loop.quit()
        elif message.type == Gst.MessageType.EOS:
            self.loop.quit()

    def run(self) -> int:
        state_result = self.pipeline.set_state(Gst.State.PLAYING)
        if state_result == Gst.StateChangeReturn.FAILURE:
            self.pipeline.set_state(Gst.State.NULL)
            raise RuntimeError("Failed to start the GStreamer pipeline")

        try:
            self.loop.run()
        finally:
            self.pipeline.set_state(Gst.State.NULL)
        return self.exit_code

    def stop(self, *_args):
        self.loop.quit()


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--video", required=True, type=Path)
    parser.add_argument("--model", required=True, type=Path)
    parser.add_argument(
        "--postprocess-library",
        type=Path,
        default=Path(
            "/usr/local/share/gstdxstream/lib/"
            "libpostprocess_yolo26od.so"
        ),
    )
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    app = PeopleCounterApplication(
        args.video,
        args.model,
        args.postprocess_library,
    )
    signal.signal(signal.SIGINT, app.stop)
    signal.signal(signal.SIGTERM, app.stop)
    return app.run()


if __name__ == "__main__":
    raise SystemExit(main())


### 5.5 Verify the DX-STREAM Python environment

The application must use the DX-STREAM virtual environment because it provides the `pydxs` module and GStreamer's Python bindings.


In [ ]:
PEOPLE_COUNTER_APP = DX_STREAM_DIR / "dx_stream/apps/yolo26_people_counter.py"
DX_STREAM_PYTHON = DX_STREAM_DIR / "venv-dx_stream/bin/python"

if not DX_STREAM_PYTHON.is_file():
    raise FileNotFoundError(
        f"DX-STREAM Python environment was not found: {DX_STREAM_PYTHON}\n"
        "Build DX-STREAM before running this application."
    )

!"{DX_STREAM_PYTHON}" -m py_compile "{PEOPLE_COUNTER_APP}"
!"{DX_STREAM_PYTHON}" -c "import gi, pydxs; print('gi and pydxs: OK')"


### 5.6 Run the application

The cell opens a GStreamer display window and runs until the video reaches EOS. Stop it with the Notebook stop button if necessary.

If a display window does not open from a Notebook cell, use **File → New → Terminal** and run the equivalent command shown below. The application must use `venv-dx_stream/bin/python`, not the Jupyter kernel, because `pydxs` and the system GStreamer bindings are installed in the DX-STREAM environment.

```bash
cd <DX_STREAM_DIR>
./venv-dx_stream/bin/python dx_stream/apps/yolo26_people_counter.py \
    --model dx_stream/samples/models/yolo26-s_640x640.dxnn \
    --video dx_stream/samples/videos/snowboard.mp4
```


In [ ]:
!"{DX_STREAM_PYTHON}" "{PEOPLE_COUNTER_APP}" \
    --model "{YOLO26S_MODEL}" \
    --video "{PEOPLE_VIDEO}"


### 5.7 Expected result and extension points

The normal YOLO boxes and labels remain visible through `dxosd`. A standard GStreamer text overlay adds the application state:

- upper right: `People: N`, updated whenever the count changes.

The displayed number should match the accepted `person` boxes for that frame. If one person produces overlapping boxes, change the postprocessor's confidence/NMS policy rather than correcting the number in the Python overlay.

For a production occupancy system, add `dxtracker`, define entry/exit regions, and count stable track IDs rather than raw per-frame detections.


## 6. Debugging

### 6.1 Use targeted GStreamer logs

Start with level 3 for initialization and state changes, then enable level 4 or 5 only for the element being investigated.

```bash
# All DX-STREAM elements at INFO level
GST_DEBUG=dx*:3 ./run_demo.sh

# Inference and tracker details
GST_DEBUG=dxinfer:4,dxtracker:4 ./run_demo.sh

# Metadata lifecycle
GST_DEBUG=dxmeta:5 ./run_demo.sh

# Save logs to a file
GST_DEBUG=2,dxinfer:4 GST_DEBUG_FILE=/tmp/dxstream.log ./run_demo.sh
```

Disable verbose logging during performance measurements because log output affects timing.


### 6.2 Understand v3.1.x failure behavior

- Incompatible links fail during caps negotiation.
- Missing model files, NPU initialization failures, and custom-library errors are reported as GStreamer element errors instead of terminating with `abort()`.
- Processing elements contribute their measured time to LATENCY queries.
- Internal state is reset on FLUSH, improving seek and replay behavior.
- EOS, FLUSH, and state transitions clean up worker threads more reliably, including single-frame input.

When a multi-stream pipeline cannot link, first check whether a standard `video/x-raw` element was placed inside the `application/x-dxvideoraw` domain.


### 6.3 Export a pipeline graph

Graphviz is required. If it is not installed, run this command in a terminal:

```bash
sudo apt install graphviz
```

The next cell stores DOT files in a dedicated directory without deleting existing reports. Stop the demo after the pipeline reaches the PLAYING state.


In [ ]:
from pathlib import Path

dot_dir = Path("tutorial_dot")
dot_dir.mkdir(exist_ok=True)

%env GST_DEBUG_DUMP_DOT_DIR=$dot_dir
!./run_demo.sh <<< 0


In [ ]:
from pathlib import Path

dot_files = sorted(Path("tutorial_dot").glob("*.dot"), key=lambda path: path.stat().st_mtime)
if not dot_files:
    raise FileNotFoundError("No DOT file was generated. Run the previous cell first.")

for dot_file in dot_files:
    print(dot_file)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

playing_files = sorted(
    Path("tutorial_dot").glob("*PAUSED_PLAYING.dot"),
    key=lambda path: path.stat().st_mtime,
)
if not playing_files:
    raise FileNotFoundError("No PAUSED_PLAYING DOT file was found.")

latest_dot = playing_files[-1]
pipeline_png = Path("tutorial_dot/PAUSED_PLAYING.png")
!dot -Tpng -o "$pipeline_png" "$latest_dot"

display(Image(filename=str(pipeline_png)))


## 7. Summary

You have now:

- Built and inspected an end-to-end DX-STREAM inference pipeline
- Ran object-detection, PPU, face-detection, pose, segmentation, tracking, multi-channel, RTSP, and Secondary Mode examples
- Inspected inference backends, `dxscale`, `dxconvert`, and the MQTT/Kafka message-broker examples
- Learned the `application/x-dxvideoraw` multi-stream domain and its element placement rules
- Adapted the YOLOv7 postprocessor and inference configuration for a two-class Forklift and Worker model
- Followed detection results from `GstBuffer` through `DXFrameMeta` and `DXObjectMeta` into Python through `pydxs`
- Embedded a GStreamer pipeline in Python and used a pad probe with the GLib main loop to display a per-frame people count
- Used targeted GStreamer logs and DOT graphs to diagnose pipeline behavior
